## 1.0 Libraries and directories

In [33]:
import ee 
import geemap
import geopandas as gpd
import datetime as dt
import pprint as pp
from shapely.geometry import shape

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

roi_name = 'YKF_sub1'
#level = 'SR' #'SR' for surface reflectance 'TOA' for Top of Atmosphere
cloud_threshold = 20

""" 
To select coincident images from a specific seasons, 
make list of timeframes with begining and end.
"""
season_start = '05-20' # 'MM-DD'
season_stop = '08-30'
year_start = 2015 # YYYY
year_stop = 2025
if year_start != year_stop:
    year_range = range(year_start, year_stop)
else:
    year_range = [year_start]

timeframes = []
for year in year_range:
    begin = str(year) + '-' + season_start
    end = str(year) + '-' + season_stop
    tf = (begin, end)
    timeframes.append(tf)

## 2.0 Convert ROI to Earth Engine Polygon

In [34]:
roi = gpd.read_file(f'./data/roi_shapes/{roi_name}_shape.shp')
#Just get the first geometry
geom = roi.geometry.iloc[0] 
coords = list(geom.exterior.coords)
coords_list = [[x, y] for x, y in coords]
roi = ee.Geometry.Polygon(coords_list)


In [38]:
def find_img_pairs(roi, start, end, cloud_threshold):
    """
    """
    s2_string = 'COPERNICUS/S2' # L1C data
    ls8_string = 'LANDSAT/LC08/C02/T1_L2'

    s2_col = (
        ee.ImageCollection(s2_string) 
        .filterDate(start, end)
        .filterBounds(roi)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
    )

    ls8_col = (
        ee.ImageCollection(ls8_string)
        .filterDate(start, end)
        .filterBounds(roi)
        .filter(ee.Filter.lt('CLOUD_COVER', cloud_threshold))
    )

    # def get_earliest_date(image_collection, collection_name):
    #     # Filter out images that have no system:time_start
    #     image_collection = image_collection.filter(ee.Filter.notNull(['system:time_start']))
        
    #     # Check size
    #     size = image_collection.size().getInfo()
    #     print(f"{collection_name} collection size after filtering null timestamps:", size)
    #     if size == 0:
    #         print(f"No valid images in {collection_name} after filtering.")
    #         return
        
    #     # Sort & get earliest image
    #     sorted_col = image_collection.sort('system:time_start')
    #     earliest_image = sorted_col.first()
        
    #     # Get the date
    #     earliest_time = earliest_image.get('system:time_start')
    #     if earliest_time is None:
    #         print(f"Earliest image in {collection_name} missing 'system:time_start' (unexpected).")
    #         return
        
    #     # Convert and print
    #     earliest_date = ee.Date(earliest_time).format('YYYY-MM-dd')
    #     date_str = earliest_date.getInfo()
    #     print(f'Earliest date in {collection_name}: {date_str}')
    
    # # Print the earliest dates
    # get_earliest_date(s2_col, 'Sentinel-2')
    # get_earliest_date(ls8_col, 'Landsat8')

    def add_date(img):
        return img.set('formatted_date', img.date().format('YYYYMMdd'))

    s2_col = s2_col.map(add_date)
    ls8_col = ls8_col.map(add_date)
    date_filter = ee.Filter.equals(leftField='formatted_date', rightField='formatted_date')
    inner_join = ee.Join.inner()
    paired_collection = inner_join.apply(s2_col, ls8_col, date_filter)

    return paired_collection

def separate_satellites(paired_fc):
    """
    Makes two sepperate collections from the combined collections
    """
    s2_images = paired_fc.map(
        lambda feature: (
            ee.Image(feature.get('primary'))
                .set('formatted_date', ee.Image(feature.get('primary')).get('formatted_date'))
        )
    )
    s2_col = ee.ImageCollection(s2_images)

    ls8_images = paired_fc.map(
        lambda feature: (
            ee.Image(feature.get('secondary'))
                .set('formatted_date', ee.Image(feature.get('secondary')).get('formatted_date'))
        )
    )
    ls8_col = ee.ImageCollection(ls8_images)

    return s2_col, ls8_col

def mosiac_attrs_by_date(img_col, roi, satellite):
    """
    For the image collection on a given satellite (Sentinel-2 or Landsat8),
    returns a summary of each unique date and the corresponding footprint (square meters) for each date
    """
    if satellite == 'Sentinel-2':
        scale = 10
        select_band = 'B8'
    else:
        scale = 30
        select_band = 'SR_B5'
    
    distinct_dates = img_col.aggregate_array('formatted_date').distinct()

    def date_mosaic(date_str):
        date_str = ee.String(date_str)
        date_imgs = img_col.filter(ee.Filter.eq('formatted_date', date_str))
        mosaic = (date_imgs.mosaic()
                  .set('fomatted_date', date_str)
                  .clip(roi)
                  .select(select_band))
        
        data_mask = mosaic.gt(0)
        polygon_boundaries = data_mask.reduceToVectors(
            #geometry=roi,
            geometryType='polygon',
            scale=scale,
            maxPixels=1e13,
            eightConnected=True
        )

        polygons_with_area = polygon_boundaries.map(lambda f: f.set({
            'area_m2': f.geometry().area(maxError=1)
        }))
        largest = polygons_with_area.sort('area_m2', False).first()

        final_feature = ee.Feature(largest).set('formatted_date', date_str)
        
        return final_feature

    footprints_fc = ee.FeatureCollection(distinct_dates.map(date_mosaic))
   
    return footprints_fc
    

In [39]:
all_imgs_list = []

for tf in timeframes:
    start = tf[0]
    end = tf[1]
    print(f'---------  Observations for {tf}  ----------')
    paired = find_img_pairs(roi, start, end, cloud_threshold=cloud_threshold)
    all_imgs_list.append(paired)

all_imgs = all_imgs_list[0]
for i in range(1, len(all_imgs_list)):
    all_imgs = all_imgs.merge(all_imgs_list[i])

s2, ls8 = separate_satellites(paired_fc=all_imgs)

#print(s2.size().getInfo())

---------  Observations for ('2015-05-20', '2015-08-30')  ----------
Sentinel-2 collection size after filtering null timestamps: 0
No valid images in Sentinel-2 after filtering.
Landsat8 collection size after filtering null timestamps: 17
Earliest date in Landsat8: 2015-05-23
---------  Observations for ('2016-05-20', '2016-08-30')  ----------
Sentinel-2 collection size after filtering null timestamps: 28
Earliest date in Sentinel-2: 2016-05-29
Landsat8 collection size after filtering null timestamps: 15
Earliest date in Landsat8: 2016-05-30
---------  Observations for ('2017-05-20', '2017-08-30')  ----------
Sentinel-2 collection size after filtering null timestamps: 28
Earliest date in Sentinel-2: 2017-05-31
Landsat8 collection size after filtering null timestamps: 13
Earliest date in Landsat8: 2017-06-02
---------  Observations for ('2018-05-20', '2018-08-30')  ----------
Sentinel-2 collection size after filtering null timestamps: 42
Earliest date in Sentinel-2: 2018-05-26
Landsat8 

In [40]:
footprints_s2 = mosiac_attrs_by_date(s2, roi=roi, satellite='Sentinel-2')
footprints_ls8 = mosiac_attrs_by_date(ls8, roi=roi, satellite='Landsat8')

In [43]:
task_s2 = ee.batch.Export.table.toDrive(
    collection=footprints_s2,
    description=f'footprints_s2_roi_{roi_name}_years_{year_start}_{year_stop}_dates_{season_start}_{season_stop}',
    folder='s2_roi_img_footprints',            
    fileNamePrefix=f'footprints_s2_roi_{roi_name}_years_{year_start}_{year_stop}_dates_{season_start}_{season_stop}',      
    fileFormat='SHP',
)
task_s2.start()

In [42]:
task_ls8 = ee.batch.Export.table.toDrive(
    collection=footprints_ls8,
    description=f'footprints_ls8_roi_{roi_name}_years_{year_start}_{year_stop}_dates_{season_start}_{season_stop}',
    folder='ls8_roi_img_footprints',            
    fileNamePrefix=f'footprints_ls8_roi_{roi_name}_years_{year_start}_{year_stop}_dates_{season_start}_{season_stop}',      
    fileFormat='SHP',
)
task_ls8.start()